## Part 1(2019 Dataset): Comparing Corpus between New Script vs. Existing Dataset

This notebook compares the corpus extracted from this repository's script to Convokit's pre-existing Supreme Court Dataset. It compares the 2019 corpus specifically, and explains the reasons for any differences that arise between the two datasets.

In [1]:
from convokit import Corpus, download
import pandas as pd
import os.path

# Pre-existing Convokit corpus
convokit_corpus = Corpus(filename = download("supreme-2019"))
# Corpus extracted from new script
new_corpus = Corpus(filename = os.path.join(os.path.expanduser("~"), ".convokit/saved-corpora/supreme-2019"))

Dataset already exists at /Users/jeeyonkang/.convokit/downloads/supreme-2019


In [2]:
convokit_2019_cases_file = './2019-cases.jsonl'
new_2019_cases_file = './new_2019_case_info.jsonl'

### Comparing cases

Among the Supreme Court cases of the 2019 term, Convokit's [file containing case information](https://zissou.infosci.cornell.edu/convokit/datasets/supreme-corpus/cases.jsonl) provides 61 cases. The new script extracts 60 cases to incorporate into the corpus. The following demonstrates which cases are included and discluded from each corpus:

In [3]:
convokit_cases_df = pd.read_json(convokit_2019_cases_file, lines=True)
new_cases_df = pd.read_json(new_2019_cases_file, lines=True)

cases_missing_from_new = new_cases_df[~new_cases_df['id'].isin(convokit_cases_df['id'])]
cases_missing_from_convokit = convokit_cases_df[~convokit_cases_df['id'].isin(new_cases_df['id'])]

print("Cases missing from Convokit (In New, not in Convokit): ", cases_missing_from_new['id'])
print()
print("Cases missing from New (In Convokit, not in New): ", cases_missing_from_convokit['id'])

Cases missing from Convokit (In New, not in Convokit):  17     2019_18-588
42    2019_18-1587
Name: id, dtype: object

Cases missing from New (In Convokit, not in New):  33    2019_18-217
52    2019_19-122
60    2019_19-373
Name: id, dtype: object


**Cases in new corpus, not in Convokit:**

The two cases with id `2019_18-588` and `2019_18-1587` are not in the original convokit corpus. 
When accessed through links of the format (www.oyez.org/cases/{term}/{docket_no}), the pages (https://www.oyez.org/cases/2019/18-588) and (https://www.oyez.org/cases/2019/18-1587) automatically redirect to (https://www.oyez.org/cases/2019/18-587) and (https://www.oyez.org/cases/2019/18-1584) respectively. Case information for cases `2019_18-587` and `2019_18-1584` exist in both the convokit corpus and the new corpus.


**Cases in Convokit, not in new corpus:**

Among the three cases that are in convokit but not in the new corpus, `2019_18-217` and `2019_19-373` are cases not in the Supreme Court Database. Since the new corpus is built based on the cases listed in the SCDB, it does not incorporate cases that are not listed. The other case, `2019_19-122`, is listed in the SCDB but does not have a 'dateArgument' field(nor does it have an oral argument Oyez transcript). Thus it is dropped from the new corpus.

### Comparing Conversations

When comparing the conversation-level date from each corpus, one may notice a difference in the conversation ids for the same case. We suspect this to be due to ongoing Oyez updates, and that as transcripts/convos get updated they are assigned a greater conversation id. If this hypothesis is true, it is natural for there to be a difference in the data, as the data in the new corpus was extracted more recently. To test this hypothesis, we confirm that the conversation_id's in the new corpus are always equal to or greater than that of the older convokit data.

In [4]:
convokit_convos_df = convokit_corpus.get_conversations_dataframe().reset_index()
new_convos_df = new_corpus.get_conversations_dataframe().reset_index()

merged_df = pd.merge(new_convos_df, convokit_convos_df, on='meta.case_id', suffixes=('_new', '_convokit'))
select_columns = ['id_new', 'id_convokit', 'meta.case_id']
print(merged_df[select_columns])

   id_new id_convokit  meta.case_id
0   25079       25048  2019_18-6135
1   25078       25047  2019_18-5924
2   25080       25050   2019_18-801
3   25053       25053  2019_17-1618
4   25081       25036   2019_18-107
5   25082       25052  2019_18-1334
6   25084       25044   2019_18-328
7   25083       25032   2019_17-834
8   25086       25049   2019_18-725
9   25087       25045   2019_18-556
10  25194       24929   2019_18-877
11  25179       24930   2019_18-565
12  24928       24928  2019_18-1165
13  24927       24927   2019_18-260
14  25164       24937  2019_17-1678
15  25178       24935   2019_18-938
16  25088       25039  2019_18-1171
17  24949       24949  2019_18-1150
18  25089       25043   2019_18-280
19  24948       24948  2019_17-1498
20  24941       24941  2019_18-1269
21  24944       24944  2019_18-1116
22  24943       24943  2019_18-6943
23  25180       24954   2019_18-916
24  24955       24955   2019_18-776
25  24957       24957  2019_18-7739
26  24952       24952  2019_

In [5]:
for id, row in merged_df.iterrows():
    assert int(row['id_new']) >= int(row['id_convokit'])

### Comparing speakers

In [6]:
convokit_speakers_df = convokit_corpus.get_speakers_dataframe().reset_index()
new_speakers_df = new_corpus.get_speakers_dataframe().reset_index()

merged_df = new_speakers_df.merge(convokit_speakers_df[['id']], on='id', how='left', indicator=True)
missing_speakers_from_new = merged_df[merged_df['_merge'] == 'left_only'].drop(columns=['_merge'])

merged_df2 = convokit_speakers_df.merge(new_speakers_df[['id']], on='id', how='left', indicator=True)
missing_speakers_from_convokit = merged_df2[merged_df2['_merge'] == 'left_only'].drop(columns=['_merge'])

# In new corpus, not in Convokit
print(f"Number of missing speakers (in new corpus, not in Convokit): {len(missing_speakers_from_new)}")
print(missing_speakers_from_new['id'])


Number of missing speakers (in new corpus, not in Convokit): 3
11       jeffrey_fisher
40     jonathan_y_ellis
64    benjamin_w_snyder
Name: id, dtype: object


In [7]:
# In Convokit, not in new corpus
print(f"Number of missing speakers (in Convokit, not in new corpus): {len(missing_speakers_from_convokit)}")
print(missing_speakers_from_convokit['id'])

Number of missing speakers (in Convokit, not in new corpus): 5
17      benjamin_snyder
45       jonathan_ellis
71       eric_a_burgess
76       toby_j_heytens
77    danielle_spinelli
Name: id, dtype: object


**Speakers in new corpus, not in previous Convokit**

Jeffrey L. Fisher's speaker id is extracted as `jeffrey_fisher` in the new corpus, because in some transcripts he appears as "Jeffrey Fisher" instead of "Jeffrey L. Fisher". His Oyez id is `jeffrey_l_fisher`. (The id `jeffrey_l_fisher` also exists in the new corpus, because in other transcripts his middle initial is included. So there are two ids for Jeffrey Fisher in the new corpus, one correct and one incorrect.)

Jonathan Y. Ellis, on the other hand, has his Oyez id as `jonathan_ellis`, but the new corpus infers his id to be `jonathan_y_ellis`. These discrepancies rise due to the speakers appearing with/without their middle name in certain transcripts. Since the code infers their Oyez id's according to the names on the transcripts, if the transcript adds or deletes a middle name, an incorrect id may arise.

On the other hand, the new corpus extracts Benjamin Snyder's id as `benjamin_w_snyder`. His id correctly exists in the new corpus, as opposed to his id being noted as `benjamin_snyder` in the previous Convokit dataset.


**Speakers in previous Convokit, not in new corpus**

As mentioned above, the id `benjamin_snyder` correctly exists as `benjamin_w_snyder` in the new corpus. 

Also mentioned above, the id `jonathan_ellis` incorrectly exists as `jonathan_y_ellis` in the new corpus.

`eric_a_burgess` is mentioned in the Convokit dataset as an advocate in case `2019_18-8379(Lomax v. Ortiz-Marquez)`. However, the advocates's name is Brian T. Burgess(id is `brian_t_burgess`, which is in both the new corpus and the pre-existing convokit corpus. Again, Brian Burgess argued in multiple cases, and in other cases his name and id were correctly extracted.)

The speakers Toby Heytens(`toby_j_heytens`) and Danielle Spinelli(`danielle_spinelli`) were dropped from the new corpus because they were advocates in case `2019_18-217`, which was dropped due to its absence from the SCDB.

### Comparing utterances

We randomly sample 30 cases, and compare the number of utterances, for each case, between the new corpus and the convokit corpus. We assert that each corpus has the same number of utterances for the same case, then assert that the content(text) of each utterance is the same as well. 

In [8]:
import random

In [9]:
convokit_utterances_df = convokit_corpus.get_utterances_dataframe().reset_index()
new_utterances_df = convokit_corpus.get_utterances_dataframe().reset_index()

case_ids_list = new_utterances_df['meta.case_id'].tolist()
random_case_ids = random.sample(case_ids_list, 30)

# print(random_case_ids)

In [10]:
for random_id in random_case_ids:
    convokit_mask = convokit_utterances_df['meta.case_id'] == random_id
    convokit_random_id_df = convokit_utterances_df[convokit_mask]

    new_mask = new_utterances_df['meta.case_id'] == random_id
    new_random_id_df = new_utterances_df[new_mask]
    
    try:
        assert len(convokit_random_id_df) == len(new_random_id_df)
    except:
        print(random_id)
        print("length of convokit utterances: ", len(convokit_random_id_df))
        print("length of new utterances: ", len(new_random_id_df))

In [11]:
merged_df = pd.merge(new_utterances_df, convokit_utterances_df, on='id', how='inner', suffixes=('_new', '_convokit'))
selected_columns = ['id', 'text_new', 'text_convokit']

# print(merged_df[selected_columns])
assert(merged_df['text_new'].equals(merged_df['text_convokit']))

## Part 2(2020 Dataset): Years Not in the Convokit Dataset

This part of the notebook checks information extracted from years not in the pre-existing Convokit dataset.

### Comparing Cases: To SCDB Information

We first compare the 2020-case-information extracted from the new script to the case information written in the Supreme Court database. It checks that all the cases that are marked in the SCDB file as having Oral Arguments, but are missing from the new corpus, are cases that do not have Oyez pages. These are cases that naturally cannot be incorporated into the Supreme Court Oral Arguments Dataset.

In [12]:
new_2020_cases_file = './new_2020_case_info.jsonl'
new_2020_cases_df = pd.read_json(new_2020_cases_file, lines=True)

scdb_cases_file = './SCDB_2024_01_caseCentered_Docket.csv'
scdb_df = pd.read_csv(scdb_cases_file)
mask = scdb_df['term'] == 2020
scdb_2020_df = scdb_df[mask]
scdb_2020_with_oral_args = scdb_2020_df.dropna(subset=['dateArgument'])

In [13]:
print(len(new_2020_cases_df))
print(len(scdb_2020_with_oral_args))

56
72


In [14]:
merged_df = pd.merge(new_2020_cases_df, scdb_2020_with_oral_args, left_on='docket_no', right_on='docket', how='outer', suffixes=('_new', '_scdb'), indicator = True)

only_in_scdb = merged_df[merged_df['_merge'] == 'right_only']
print([only_in_scdb['docketId']])

[4     2020-018-02
8     2020-042-02
12    2020-038-02
14    2020-046-02
18    2020-044-02
19    2020-044-03
20    2020-008-02
23    2020-056-02
29    2020-005-02
33    2020-021-02
39    2020-029-02
50    2020-047-02
66    2020-055-02
68    2020-065-02
70    2020-040-01
71    2020-001-01
Name: docketId, dtype: object]


On Nov. 21, 2024, **Manually checked that all of the above cases do not have Oyez pages**.

### Comparing Utterances

Next, we randomly sample five cases. For each case, we compare the first utterance in the new corpus to the line directly copy-and-pasted from the Oyez transcript. We affirm that the two match.

In [15]:
new_corpus_2020 = Corpus(filename = os.path.join(os.path.expanduser("~"), ".convokit/saved-corpora/supreme-2020"))
new_2020_utterances_df = new_corpus_2020.get_utterances_dataframe().reset_index()

case_ids_list_2020 = new_2020_utterances_df['meta.case_id'].tolist()
random_case_ids_2020 = random.sample(case_ids_list_2020, 5)
print(random_case_ids_2020)

['2020_19-546', '2020_19-416', '2020_19-840', '2020_20-18', '2020_19-546']


While testing on Nov. 21, 2024, the above output gave ['2020_20-543', '2020_20-472', '2020_20-297', '2020_18-1259', '2020_19-1434'] as random ids.

In [16]:
#These are manually copy&pasted from the Oyez page
gold_2020_20_543 = "We will hear argument first this morning in Case 20-543, Yellen versus the Confederated Tribes, and the consolidated case. Mr. Guarnieri."
gold_2020_20_472 = "We will hear argument first this morning in Case 20-472, HollyFrontier Cheyenne Refining versus Renewable Fuels Association. Mr. Keisler."
gold_2020_20_297 = "We will hear argument this morning in Case 20-297, TransUnion versus Ramirez. Mr. Clement."
gold_2020_18_1259 = "We will hear argument first this morning in Case Number 18-1259, Jones versus Mississippi. Mr. Shapiro."
gold_2020_19_1434 = "We will hear argument this morning in Case 19-1434, United States versus Arthrex, Incorporated, and the consolidated cases. Mr. Stewart."

In [17]:
random_case_ids_2020 = ['2020_20-543', '2020_20-472', '2020_20-297', '2020_18-1259', '2020_19-1434'] # as of Nov. 21, 2024
gold_list = [gold_2020_20_543, gold_2020_20_472, gold_2020_20_297, gold_2020_18_1259, gold_2020_19_1434]
gold_standards = zip(random_case_ids_2020, gold_list)
for pair in gold_standards:
    random_id = pair[0]
    gold = pair[1]
    text_column = new_2020_utterances_df[new_2020_utterances_df['meta.case_id'] == random_id]['text']
    assert text_column.iloc[0] == gold

Testing Nov. 21, 2024:

Manually confirmed that the actual first line of the transcripts of the above cases match the extracted data.